In [ ]:
import base64, html
from search_new import simple_search, smart_search, wd14_search, rewrite_query
from IPython.display import display, Image as IPImage, HTML
from pathlib import Path

In [ ]:

def _thumb(rec, max_px=200):
    """优先本地图 base64 内联(离线可用),退回 file_url。"""
    p = rec.get("local_image_path", "")
    if p and Path(p).exists():
        ext = Path(p).suffix.lstrip(".").lower() or "jpg"
        mime = "jpeg" if ext in ("jpg", "jpeg") else ext
        b64 = base64.b64encode(Path(p).read_bytes()).decode()
        src = f"data:image/{mime};base64,{b64}"
    else:
        src = rec.get("file_url", "")
    return (f'<img src="{src}" loading="lazy" '
            f'style="max-width:{max_px}px;max-height:{max_px}px;'
            f'border-radius:6px;display:block;margin:auto;">')


def _score_of(rec):
    return rec.get("_final_score", rec.get("_score", 0.0))


def _matched_str(rec):
    m = rec.get("_matched_tags")
    return "<br>".join(f"{t}: {c}" for t, c in m.items()) if m else ""


def compare_search(query, top_k=4):
    """同一句自然语言,三条检索臂并排出结果。"""
    results = {
        "simple<br><span style='color:#999'>embedding baseline</span>":
            simple_search(query, top_k=top_k),
        "smart<br><span style='color:#999'>human ∪ WD14</span>":
            smart_search(query, top_k=top_k, verbose=False),
        "wd14-only<br><span style='color:#999'>NL → WD14</span>":
            wd14_search(rewrite_query(query).split(), top_k=top_k, verbose=False),
    }
    cols = list(results.keys())
    n = max((len(v) for v in results.values()), default=0)

    th = "".join(
        f'<th style="padding:8px 12px;border-bottom:2px solid #ddd;'
        f'font-size:13px;text-align:center;">{c}</th>' for c in cols
    )
    body = ""
    for i in range(n):
        tds = ""
        for c in cols:
            r = results[c][i] if i < len(results[c]) else None
            if r is None:
                tds += '<td style="padding:8px;"></td>'
                continue
            cap = f'#{r["_rank"]} · id={r.get("id", "?")} · {_score_of(r):.3f}'
            mt = _matched_str(r)
            mt_html = (f'<div style="font-size:11px;color:#aaa;margin-top:2px;">{mt}</div>'
                       if mt else "")
            tds += (f'<td style="padding:8px;text-align:center;vertical-align:top;">'
                    f'{_thumb(r)}'
                    f'<div style="font-size:12px;margin-top:4px;color:#444;">{cap}</div>'
                    f'{mt_html}</td>')
        body += f"<tr>{tds}</tr>"

    html_out = (
        f'<div style="font-family:system-ui,sans-serif;">'
        f'<div style="font-size:15px;font-weight:600;margin-bottom:10px;">'
        f'🔍 Query: {html.escape(query)}</div>'
        f'<table style="border-collapse:collapse;margin:auto;">'
        f'<thead><tr>{th}</tr></thead><tbody>{body}</tbody></table></div>'
    )
    display(HTML(html_out))


# 用法
compare_search("带爪子的龙娘从背后看")

In [ ]:
# 用之前的 show_results,但换成 smart_search
def show_smart_results(query: str, top_k: int = 5):
    results = smart_search(query, top_k=top_k)
    
    for r in results:
        print(f"#{r['_rank']}  final={r['_final_score']:.3f}  "
              f"vec={r['_vec_sim']:.3f}  pos={r['_pos_score']:.3f}  "
              f"neg={r['_neg_score']:.3f}  id={r['id']}")
        print(f"   general: {r.get('tag_string_general', '')[:100]}")
        
        img_path = r.get("local_image_path")
        if img_path and Path(img_path).exists():
            display(IPImage(filename=img_path, width=300))
        print()
        
    
    return results

# 测试

# === 组 3: 验证 vec + tag 双信号 ===
from search_new import smart_search

# Case 1: 之前 pos=0.1,看现在能不能涨到 0.2
show_smart_results("金发女孩", top_k=5)
